# Living allocations

In [46]:

from __future__ import annotations

import os
import re
import ast
import json
import math
import hashlib
from collections import Counter
from pathlib import Path
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer, util

import matplotlib.pyplot as plt
import seaborn as sns

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

print("Imports successfully loaded!")

Imports successfully loaded!


## Loading Datasets

In [ ]:
df_all_wrv = pd.read_csv(
    r'Data\WRV_bezwaren_.csv', 
    sep=';', 
    on_bad_lines='warn',
    engine='python',
    encoding='cp1252'
)

df_all_wrv.columns = df_all_wrv.columns.str.strip()
df_wrv_training = df_all_wrv.sample(frac=0.8, random_state=42).copy()
df_wrv_testing = df_all_wrv.drop(df_wrv_training.index).copy()

possible_id_cols = ['Octopus zaaknummer', 'zaaknummer', 'Zaaknummer', 'case_id']
id_col = next((col for col in possible_id_cols if col in df_all_wrv.columns), None)

if id_col is None:
    raise KeyError(f"Could not find the ID column. Available columns are: {list(df_all_wrv.columns)}")

training_ids = df_wrv_training[id_col].tolist()
testing_ids = df_wrv_testing[id_col].tolist()

print(f"Split complete: {len(training_ids)} training cases, {len(testing_ids)} testing cases.")


def subtype(subject):
    s = str(subject).lower()
    
    if 'afbouw' in s:
        if 'alle punten' in s:
            return 'Total Point Reset'
        return 'Point Sanction (Reduction)'
    
    if 'afwijzing' in s or 'rejection' in s:
        if 'jongeren' in s:
            return 'Rejection: Youth Category'
        if 'gezin' in s:
            return 'Rejection: Family Category'
        return 'Application Rejection'

    if 'situatiepunten' in s and 'start' in s:
        return 'Allocation: Start & Situation Points'
    if 'situatiepunten' in s:
        return 'Allocation: Situation Points'
    if 'startpunten' in s:
        return 'Allocation: Starting Points'
    if 'zoekpunten' in s:
        return 'Allocation: Search Points'
    
    return 'Other / Miscellaneous'

df_wrv_training['Onderwerp'] = df_wrv_training['Onderwerp'].apply(subtype)
df_wrv_training.rename(columns={
    'Onderwerp': 'Subtype', 
    'geanonimiseerd_doc_inhoud': 'Legal_Text'
}, inplace=True)

df_wrv_testing['Onderwerp'] = df_wrv_testing['Onderwerp'].apply(subtype)
df_wrv_testing.rename(columns={
    'Onderwerp': 'Subtype', 
    'geanonimiseerd_doc_inhoud': 'Legal_Text'
}, inplace=True)


train_counts = df_wrv_training['Subtype'].value_counts()
test_counts = df_wrv_testing['Subtype'].value_counts()

subtype_comparison = pd.concat([train_counts, test_counts], axis=1)
subtype_comparison.columns = ['Training Set', 'Testing Set']
subtype_comparison = subtype_comparison.fillna(0).astype(int)

print("\n--- Subtype Comparison: Training vs Testing ---")
print(subtype_comparison)

if 'Dictum' in df_wrv_training.columns and 'Dictum' in df_wrv_testing.columns:
    dictum_comparison = pd.concat([
        df_wrv_training['Dictum'].value_counts(),
        df_wrv_testing['Dictum'].value_counts()
    ], axis=1)
    dictum_comparison.columns = ['Training Set', 'Testing Set']
    print("\n--- Dictum Comparison: Training vs Testing ---")
    print(dictum_comparison.fillna(0).astype(int))

Split complete: 743 training cases, 186 testing cases.

--- Subtype Comparison: Training vs Testing ---
                                      Training Set  Testing Set
Subtype                                                        
Point Sanction (Reduction)                     463          123
Rejection: Family Category                     197           39
Rejection: Youth Category                       34           15
Total Point Reset                               18            6
Application Rejection                           18            1
Allocation: Situation Points                     8            1
Allocation: Starting Points                      3            0
Allocation: Start & Situation Points             2            1

--- Dictum Comparison: Training vs Testing ---
                           Training Set  Testing Set
Dictum                                              
Gegrond                             358          100
Ongegrond                           335          

In [48]:
df_wrv_testing['Octopus zaaknummer'][:50]

1      WO.23.002023.001
4      WO.23.002302.001
13     WO.23.003140.001
14     WO.23.003258.001
20     WO.23.003993.001
21     WO.23.004340.001
27     WO.23.004695.001
32     WO.23.004978.001
34     WO.23.004995.001
35     WO.23.004999.001
40     WO.23.005025.001
47     WO.23.005621.001
52     WO.23.006078.001
58     WO.23.006198.001
62     WO.23.006544.001
64     WO.23.006560.001
71     WO.23.007134.001
80     WO.23.007232.001
85     WO.23.007392.001
87     WO.23.007415.001
91     WO.23.007457.001
95     WO.23.007570.001
98     WO.23.007578.001
99     WO.23.007607.001
102    WO.23.007673.001
105    WO.23.007699.001
106    WO.23.007754.001
121    WO.23.008161.001
128    WO.23.008468.001
130    WO.23.008498.001
134    WO.23.008581.001
138    WO.23.008626.001
156    WO.23.009083.001
159    WO.23.009143.001
160    WO.23.009160.001
161    WO.23.009175.001
166    WO.23.009249.001
170    WO.23.009503.001
187    WO.23.010004.001
189    WO.23.010015.001
191    WO.23.010052.001
200    WO.23.010

## Chunking

In [ ]:
HEADER_ALIASES_BY_KIND = {
    "woonruimteverdeling": {
        "chunk_bezwaar": {
            "bezwaren", "de bezwaren", "uw bezwaar", "uw bezwaren", 
            "bezwaar", "tegen het besluit", "bezwaargronden", "gronden van bezwaar", 
            "standpunt bezwaarmaker", "bezwaarschrift"
        },
        "chunk_hoorzitting": {
            "hoorzitting", "horen", "telefonisch horen", "telefonisch contact", 
            "digitale hoorzitting", "informele hoorzitting", "informeel overleg", 
            "hoorverslag", "gesprek", "telefonisch verslag"
        },
        "chunk_wetten": {
            "juridisch kader", "wet- en regelgeving", "wet en regelgeving", 
            "wettelijk kader", "toepasselijke regels", "bijlagen wettelijke kader"
        },
        "chunk_uitleg": {
            "beoordeling bezwaarschrift", "overwegingen", "motivering", 
            "beoordeling", "toetsingskader", "advies", "conclusie", 
            "slotsom", "beslissing", "aanleiding", "inleiding", "feiten", 
            "ontvankelijkheid", "procesverloop", "beslissing op bezwaar", 
            "overwegingen en beoordeling bezwaar", "dit betekent voor u", 
            "wij wijzen u verder", "overwegingen bij bezwaren en toelichting op het besluit"
        },
    }
}

IGNORE_HEADER_PATTERNS = [
    r"^met vriendelijke groet", r"^hoogachtend", r"^stuur uw beroepschrift naar", 
    r"^rechtbank", r"^directeur\b.*", r"^gemeente amsterdam\b.*", 
    r"^[^\n]*gemeente\s+amsterdam(?:\s+en\s+\w+)?[^\n]*", r"^in uw beroepschrift moet staan", 
    r"^octopus[-\s]*nummers?:", r"^zaak[-\s]*nummer", r"^ten aanzien van", 
    r"^gericht tegen", r"^ingediend door", r"^betreft:", r"^dhr\.", r"^mevr\.", r"^\[naam\]",
    r"^mevrouw\b.*", r"^meneer\b.*", r"^de heer\b.*" 
]

def _compute_md5(text: str) -> str:
    return hashlib.md5((text or "").encode("utf-8")).hexdigest()

def _normalize_text_for_hash(text: str) -> str:
    if pd.isna(text):
        return ""
    return re.sub(r"[^\S\n]+", " ", str(text)).strip()

def _detect_text_column(df: pd.DataFrame, preferred: str = "") -> str:
    if preferred and preferred in df.columns:
        return preferred
    candidates = ["geanonimiseerd_doc_inhoud", "Legal_Text", "Document inhoud", "text", "tekst"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError("No valid text column detected in DataFrame.")

def _restore_missing_newlines(text: str) -> str:
    if pd.isna(text):
        return ""
    
    flat_headers = [
        "Beslissing op bezwaar", "Overwegingen en beoordeling bezwaar",
        "Overwegingen en beoordeling", "Overwegingen bij bezwaren en toelichting op het besluit",
        "Dit betekent voor u:", "Beroep", "Vragen?", "Met vriendelijke groet,",
        "Huisvestingswet 2014", "Huisvestingsverordening Amsterdam 2020",
        "Huisvestingsverordening Amsterdam 2024", "Nadere Regels Huisvestingsverordening",
        "Algemene wet bestuursrecht", "Bezwaren", "Hoorzitting", "Telefonisch verslag",
        "Wettelijk kader", "Wet- en regelgeving", "Toepasselijke regels", "Zorgregeling" 
    ]
    
    for h in flat_headers:
        h_escaped = re.escape(h)
        text = re.sub(rf"\s*({h_escaped})\s*", r"\n\1\n", text)
        
    return re.sub(r"\n{2,}", "\n\n", text).strip()

def _normalize_header_key(line: str) -> str:
    t = (line or "").strip()
    t = re.sub(r"^(?:\d+|[IVXLC]+)[\.\)\s,-]+", "", t, flags=re.IGNORECASE)
    t = re.sub(r"[\s:;?!\.]+$", "", t)
    return re.sub(r"\s+", " ", t).strip().lower()

def _alias_map_for_kind(kind: str, section_sets: dict[str, set[str]]) -> dict[str, str]:
    alias_map: dict[str, str] = {}
    for section, names in (section_sets or {}).items():
        for name in names:
            norm = _normalize_header_key(name)
            if norm:
                alias_map[norm] = section
    return alias_map

def _is_ignored_header_line(line: str) -> bool:
    t = (line or "").strip()
    if not t:
        return True
    return any(re.match(pat, t, flags=re.IGNORECASE) for pat in IGNORE_HEADER_PATTERNS)

def _looks_like_header(line: str) -> bool:
    # Validates if a line structurally looks like a section header
    t = (line or "").strip()
    if not t:
        return False
    candidate = re.sub(r"^(?:\d+|[IVXLC]+)[\.\)\s-]+", "", t, flags=re.IGNORECASE).strip().rstrip(":").strip()
    if not candidate or len(candidate) > 120:
        return False
    
    words = re.findall(r"[A-Za-z\u00C0-\u00FF]+", candidate)
    if len(words) > 14:
        return False
    if t.endswith(":") or candidate.isupper():
        return True
    if all(w[:1].isupper() for w in words) and not re.search(r"[\.!?]", candidate):
        return True
    
    return len(words) <= 7 and len(candidate) <= 70 and not re.search(r"[\.!?]", candidate)

def _is_header_line(line: str, header_to_section: Optional[dict]) -> bool:
    if _is_ignored_header_line(line):
        return False
    if header_to_section and _normalize_header_key(line) in header_to_section:
        return True
    return _looks_like_header(line)

def _guess_section_for_header(norm: str) -> Optional[str]:
    # Fallback keyword matching for unknown headers
    if "bezwaar" in norm and "beslissing" not in norm and "beoordeling" not in norm:
        return "chunk_bezwaar"
    if any(k in norm for k in ("wet", "regel", "juridisch", "kader", "bijlage")):
        return "chunk_wetten"
    if any(k in norm for k in ("hoor", "gesprek", "telefonisch", "mondeling", "verslag")):
        return "chunk_hoorzitting"
    if any(k in norm for k in ("overweging", "beoordeling", "conclusie", "besluit", "beslissing", "betekent", "wijzen")):
        return "chunk_uitleg"
    return None

def _chunk_document_by_short_titles(document: str, header_to_section: Optional[dict] = None) -> list[str]:
    paragraphs = (document or "").split("\n")
    chunks: list[str] = []
    current_chunk: list[str] = []

    for paragraph in paragraphs:
        if paragraph.strip() == "" or _is_ignored_header_line(paragraph):
            current_chunk.append(paragraph)
            continue

        if _is_header_line(paragraph, header_to_section):
            if current_chunk:
                chunks.append("\n".join(current_chunk))
            current_chunk = [paragraph]
        else:
            current_chunk.append(paragraph)

    if current_chunk:
        chunks.append("\n".join(current_chunk))
    return chunks

def _derive_paragraphs_from_chunks(chunks: list[str], section_sets: dict[str, set[str]], fallback_key: str) -> dict:
    sections = {k: [] for k in section_sets.keys()}
    sections.setdefault("paragraaf_overig", [])
    header_to_section = _alias_map_for_kind("woonruimteverdeling", section_sets)

    for chunk in chunks:
        lines = chunk.split("\n")
        if not lines:
            continue

        header_idx = next((i for i, v in enumerate(lines) if v.strip() and not _is_ignored_header_line(v)), None)
        if header_idx is None:
            continue

        header_line = lines[header_idx]
        header = _normalize_header_key(header_line)

        if _is_header_line(header_line, header_to_section):
            target = header_to_section.get(header) or _guess_section_for_header(header) or fallback_key
        else:
            if "maakte u bezwaar" in header_line.lower() or "tegen het besluit" in header_line.lower():
                target = "chunk_bezwaar"
            else:
                target = fallback_key

        content = "\n".join(lines)
        if content.strip():
            sections[target].append(content)

    result_dict = {"chunk_bezwaar": "", "chunk_hoorzitting": "", "chunk_uitleg": "", "chunk_wetten": ""}
    
    result_dict["chunk_bezwaar"] = "\n".join([s for s in sections.get("chunk_bezwaar", []) if s]).strip()
    result_dict["chunk_uitleg"] = "\n".join([s for s in sections.get("chunk_uitleg", []) + sections.get("paragraaf_overig", []) if s]).strip()
    result_dict["chunk_hoorzitting"] = "\n".join([s for s in sections.get("chunk_hoorzitting", []) if s]).strip()
    result_dict["chunk_wetten"] = "\n".join([s for s in sections.get("chunk_wetten", []) if s]).strip()

    return result_dict

def _derive_paragraphs_from_text(text: str, section_sets: dict[str, set[str]], fallback_key: str) -> dict:
    header_to_section = _alias_map_for_kind("woonruimteverdeling", section_sets)
    chunks = _chunk_document_by_short_titles(text or "", header_to_section=header_to_section)
    return _derive_paragraphs_from_chunks(chunks, section_sets, fallback_key)

def chunk_housing_dataframe(
    df: pd.DataFrame,
    text_column: str = "",
    dedupe: bool = True,
    output_path: Optional[str | Path] = None
) -> pd.DataFrame:
    """Main execution logic to chunk legal text into predefined columns."""
    df_processed = df.copy()
    
    source_col = _detect_text_column(df_processed, preferred=text_column)
    
    df_processed["geanonimiseerd_doc_inhoud"] = df_processed[source_col].fillna("").apply(_restore_missing_newlines)

    if dedupe:
        df_processed["__doc_hash"] = df_processed["geanonimiseerd_doc_inhoud"].apply(_normalize_text_for_hash).apply(_compute_md5)
        df_processed = df_processed.drop_duplicates(subset=["__doc_hash"], keep="first")
        df_processed = df_processed.drop(columns=["__doc_hash"])

    section_sets = HEADER_ALIASES_BY_KIND["woonruimteverdeling"]
    fallback_key = "chunk_uitleg"

    derived = df_processed["geanonimiseerd_doc_inhoud"].apply(
        lambda t: _derive_paragraphs_from_text(t, section_sets, fallback_key)
    ).apply(pd.Series)

    for c in derived.columns:
        if c.startswith("chunk_"):
            df_processed[c] = derived[c].fillna("")

    return df_processed


print("Chunking training data (df_wrv_training)...")
df_wrv_training = chunk_housing_dataframe(df_wrv_training, text_column="Legal_Text", dedupe=False)

print("Chunking testing data (df_wrv_testing)...")
df_wrv_testing = chunk_housing_dataframe(df_wrv_testing, text_column="Legal_Text", dedupe=False)

print("Chunking complete for both df_wrv_training and df_wrv_testing!")

Chunking training data (df_wrv_training)...
Chunking testing data (df_wrv_testing)...
Chunking complete for both df_wrv_training and df_wrv_testing!


In [ ]:

print(df_wrv_testing.head())

df_wrv_testing.to_csv('Data_created/wrv_testing_v1.csv') 

df_wrv_training.to_csv('Data_created/wrv_training_v1.csv') 

   Octopus zaaknummer Zaaktype                     Subtype Primair Besluit  \
1    WO.23.002023.001  Bezwaar  Rejection: Family Category   GRP-SIT-32329   
4    WO.23.002302.001  Bezwaar  Rejection: Family Category   GRP-SIT-32225   
13   WO.23.003140.001  Bezwaar  Rejection: Family Category   GRP-SIT-32122   
14   WO.23.003258.001  Bezwaar   Rejection: Youth Category   GRP-STP-33056   
20   WO.23.003993.001  Bezwaar  Point Sanction (Reduction)     F2023.18114   

   Verzenddatum primair besluit                     Dictum   Zaakstatus  \
1                     27-1-2023                    Gegrond  Afgehandeld   
4                     27-1-2023                  Ongegrond  Afgehandeld   
13                     7-2-2023                  Ongegrond  Afgehandeld   
14                     9-2-2023                  Ongegrond  Afgehandeld   
20                    10-2-2023  Gegrond - terugverwijzing  Afgehandeld   

              Behandelaar Startdatum Datum zitting Verzendatum besluit  \
1   Ka

# Test

In [75]:
case_id_test = "WO.23.004999.001"

## RAG

In [ ]:
for df in [df_wrv_training, df_wrv_testing]:
    df.rename(columns={"Octopus zaaknummer": "case_id", "Dictum": "dictum"}, inplace=True)

df_wrv_training["dictum"] = df_wrv_training["dictum"].fillna("").str.lower()
df_wrv_testing["dictum"] = df_wrv_testing["dictum"].fillna("").str.lower()

In [ ]:
_TOKEN_RE = re.compile(r"[a-z0-9]+", flags=re.IGNORECASE)
_STOPWORDS = {
    "de", "het", "een", "en", "van", "te", "dat", "die", "dit", "in", "op", "met", 
    "voor", "door", "is", "zijn", "aan", "als", "om", "bij", "u", "uw", "wij", 
    "ons", "onze", "ik", "hij", "zij", "er", "niet", "wel", "dan"
}

def _tokenize(text: str) -> List[str]:
    safe_text = str(text) if pd.notna(text) else ""
    tokens = [token.lower() for token in _TOKEN_RE.findall(safe_text)]
    return [token for token in tokens if token not in _STOPWORDS]

def _l2_normalize(vector: Dict[str, float]) -> Dict[str, float]:
    norm = math.sqrt(sum(weight * weight for weight in vector.values()))
    return {token: (weight / norm) for token, weight in vector.items()} if norm > 0 else {}

class SimpleVectorRag:
    """Vector-RAG using TF-IDF and cosine similarity."""

    def __init__(self, cases: pd.DataFrame):
        self.cases = cases.reset_index(drop=True).copy()
        
        # Build index from Legal_Text
        self._doc_tokens = [_tokenize(text) for text in self.cases["Legal_Text"].tolist()]
        self._idf = self._build_idf()
        self._doc_vectors = [self._build_tfidf_vector(Counter(t)) for t in self._doc_tokens]

    def _build_idf(self) -> Dict[str, float]:
        doc_count = len(self._doc_tokens)
        if doc_count == 0: return {}
        
        df_counts = Counter()
        for tokens in self._doc_tokens:
            df_counts.update(set(tokens))
            
        return {t: math.log((doc_count + 1.0) / (c + 1.0)) + 1.0 for t, c in df_counts.items()}

    def _build_tfidf_vector(self, term_freq: Counter) -> Dict[str, float]:
        total = sum(term_freq.values())
        if total <= 0: return {}
        
        vector = {}
        for token, count in term_freq.items():
            idf = self._idf.get(token, 0.0)
            if idf > 0:
                vector[token] = (count / total) * idf
        return _l2_normalize(vector)

    def _cosine_score(self, query_vector: Dict[str, float], index: int) -> float:
        doc_vector = self._doc_vectors[index]
        return sum(weight * doc_vector.get(token, 0.0) for token, weight in query_vector.items())

    def retrieve(self, query: str, *, top_k: int, dictum: Optional[str] = None) -> List[dict]:
        query_vector = self._build_tfidf_vector(Counter(_tokenize(query)))
        candidate_indices = list(range(len(self.cases)))

        target_dictum = (dictum or "").strip().lower()
        if target_dictum in {"gegrond", "ongegrond"}:
            candidate_indices = [
                i for i in candidate_indices 
                if str(self.cases.iloc[i].get("dictum", "")).strip().lower() == target_dictum
            ]

        scored = [(i, self._cosine_score(query_vector, i)) for i in candidate_indices]
        scored.sort(key=lambda x: x[1], reverse=True)

        results = []
        for index, score in scored[:max(top_k, 1)]:
            row = self.cases.iloc[index]
            results.append({
                "case_id": row.get("case_id", ""),
                "dictum": row.get("dictum", ""),
                "score": float(score),
                "Legal_Text": row.get("Legal_Text", ""),
                "chunk_bezwaar": row.get("chunk_bezwaar", ""),
                "chunk_uitleg": row.get("chunk_uitleg", ""),
                "chunk_hoorzitting": row.get("chunk_hoorzitting", ""),
                "chunk_wetten": row.get("chunk_wetten", ""),
            })
        return results

In [ ]:
# --- 4. FIND SIMILAR CASES BASED ON A TARGET CASE ---

def find_similar_cases(target_case_id: str, rag_system: SimpleVectorRag, query_df: pd.DataFrame, top_k: int = 3):
    target_row = query_df[query_df['case_id'] == target_case_id]
    
    if target_row.empty:
        print(f"Error: Case ID '{target_case_id}' not found.")
        return
        
    target_text = str(target_row.iloc[0].get('Legal_Text', '')).strip()
    
    if not target_text:
        print(f"Error: Case ID '{target_case_id}' has empty text.")
        return

    print(f"Finding training cases similar to test case: {target_case_id}...\n")
    
    results = rag_system.retrieve(query=target_text, top_k=top_k)
    
    match_count = 0
    for res in results:
        if res['case_id'] == target_case_id:
            continue
            
        if res['score'] == 0.0:
            continue
            
        match_count += 1
        doc_word_count = len(str(res.get('Legal_Text', '')).split())
            
        print(f"===================== MATCH {match_count} =====================")
        print(f"Similar Case ID: {res['case_id']}")
        print(f"Match Score:     {res['score']:.4f}")
        print(f"Doc Length:      ~{doc_word_count} words")
        print(f"Dictum:          {res['dictum']}")
        print("-" * 40)
        
        full_text = str(res.get('Legal_Text', ''))
        print("TEXT PREVIEW:")
        print(full_text[:500] + ("..." if len(full_text) > 500 else ""))
        print("=" * 60 + "\n")
        
        if match_count >= top_k:
            break
            
    if match_count == 0:
        print("No similar cases found.")


rag_system = SimpleVectorRag(cases=df_wrv_training)
target_case_test = case_id_test

find_similar_cases(
    target_case_id=target_case_test, 
    rag_system=rag_system, 
    query_df=df_wrv_testing,
    top_k=3
)

Finding training cases similar to test case: WO.23.004999.001...

===================== MATCH 1 =====================
Similar Case ID: WO.23.012578.001
Match Score:     0.5308
Doc Length:      ~1359 words
Dictum:          ongegrond
----------------------------------------
TEXT PREVIEW:
Op 14 juli 2023 maakte u bezwaar tegen het besluit van 9 juni 2023, waarbij uw aanvraag voor
startpunten is afgewezen op grond van artikel 2.7.2 onder b van de Huisvestingsverordening
Amsterdam 2020 (hierna: HVV). Reden hiervoor is, dat u op datum van de aanvraag niet vier jaar
en zes maanden ingeschreven staat op het adres van de jongerenwoning.

Beslissing op bezwaar
Wij verklaren uw bezwaarschrift ongegrond. Het bestreden besluit wordt daarmee in stand
gelaten. Dit betekent dat u geen gelijk ...

===================== MATCH 2 =====================
Similar Case ID: WO.23.002887.001
Match Score:     0.5268
Doc Length:      ~529 words
Dictum:          gegrond
----------------------------------------
TEXT

## Legal Bert

In [ ]:

for df in [df_wrv_training, df_wrv_testing]:
    df.rename(columns={"Octopus zaaknummer": "case_id", "Legal_Text": "Legal_Text"}, errors="ignore", inplace=True)

test_case_ids = set(df_wrv_testing["case_id"].dropna().unique())
df_train_clean = df_wrv_training[~df_wrv_training["case_id"].isin(test_case_ids)].copy()

print(f"Data Leakage Check: Removed {len(df_wrv_training) - len(df_train_clean)} overlapping cases.\n")

class LegalBertRag:
    def __init__(self, df: pd.DataFrame, model_name: str = "nlpaueb/legal-bert-base-uncased"):
        self.cases = df.reset_index(drop=True).copy()
        
        os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading {model_name} on {self.device}...")
        self.model = SentenceTransformer(model_name, device=self.device)
        print(f"Encoding {len(self.cases)} documents...")
        texts = self.cases["chunk_bezwaar"].fillna("").astype(str).tolist()
        self.corpus_embeddings = self.model.encode(texts, convert_to_tensor=True, show_progress_bar=True)

    def retrieve(self, query: str, top_k: int = 3, dictum: Optional[str] = None) -> List[dict]:
        query_embedding = self.model.encode(query, convert_to_tensor=True)
        cos_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0]
        valid_indices = torch.arange(len(self.cases), device=self.device)
        
        if dictum:
            target_dictum = str(dictum).strip().lower()
            dictum_col = "dictum" if "dictum" in self.cases.columns else "Dictum"
            mask = [str(d).strip().lower() == target_dictum for d in self.cases[dictum_col]]
            valid_indices = valid_indices[torch.tensor(mask, device=self.device)]
            
            if len(valid_indices) == 0:
                return []
            cos_scores = cos_scores[valid_indices]
            
        top_k = min(top_k, len(valid_indices))
        top_results = torch.topk(cos_scores, k=top_k)
        
        results = []
        for score, local_idx in zip(top_results[0], top_results[1]):
            actual_idx = valid_indices[local_idx].item()
            row = self.cases.iloc[actual_idx]
            
            results.append({
                "case_id": row.get("case_id", "Unknown"),
                "score": score.item(),
                "chunk_wetten": row.get("chunk_wetten", ""),
                "chunk_uitleg": row.get("chunk_uitleg", ""),
                "target_text": row.get("Legal_Text", "")
            })
        return results
print("\n--- Re-initializing RAG System with dictum support ---")
rag_system_bert = LegalBertRag(df=df_train_clean)


Data Leakage Check: Removed 0 overlapping cases.


--- Re-initializing RAG System with dictum support ---
Loading nlpaueb/legal-bert-base-uncased on cpu...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 38868.70it/s]
[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding 743 documents...


Batches:   0%|          | 0/24 [00:03<?, ?it/s]


KeyboardInterrupt: 

In [ ]:

def get_similar_cases_bert(case_id: str, rag_instance: LegalBertRag, test_df: pd.DataFrame, top_k: int = 3):
    """Fetches a test case by ID and retrieves similar training cases instantly."""
    case_row = test_df[test_df["case_id"] == case_id]
    if case_row.empty:
        print(f"Error: Case ID '{case_id}' not found in the test set.")
        return
    query_text = str(case_row.iloc[0].get("Legal_Text", ""))
    if not query_text.strip():
        print(f"Error: Legal_Text for '{case_id}' is empty.")
        return 
    print(f"\nSearching training database for cases similar to Test Case: {case_id}...\n")
    
    # Retrieve using the pre-loaded instance
    search_results = rag_instance.retrieve(query=query_text, top_k=top_k)
    
    for rank, res in enumerate(search_results, 1):
        print(f"Rank {rank}: Score = {res['score']:.4f} | Case ID: {res['case_id']}")
        # Print a snippet of 200 characters for readability
        snippet = res['target_text'][:200].replace('\n', ' ')
        print(f"Snippet: {snippet}...\n")
        print("-" * 50)


get_similar_cases_bert(case_id=case_id_test, rag_instance=rag_system_bert, test_df=df_wrv_testing, top_k=3)


Searching training database for cases similar to Test Case: WO.23.004999.001...

Rank 1: Score = 0.9889 | Case ID: WO.23.012997.001
Snippet: Op 26 juli 2023 maakte u bezwaar tegen het besluit van 14 juli 2023, waarbij uw aanvraag voor situatiepunten is afgewezen op grond van artikel 2.5.2 lid 3 onder b uit de Huisvestingsverordening Amster...

--------------------------------------------------
Rank 2: Score = 0.9889 | Case ID: WO.24.002425.001
Snippet: Op 16 februari 2024 maakte u bezwaar tegen het besluit van 8 februari 2024, waarbij uw aanvraag voor situatiepunten is afgewezen op grond van artikel 2.6.2 lid 3 onder a en b uit de Huisvestingsverord...

--------------------------------------------------
Rank 3: Score = 0.9888 | Case ID: WO.23.004374.001
Snippet: Op 20 februari 2023 maakte u bezwaar tegen het besluit van 20 februari 2023 waarbij uw aanvraag voor situatiepunten is afgewezen op grond van artikel 2.5.2 lid 3b uit de Huisvestingsverordening 2020 A...

---------------------

: 

## LLM + RAG

In [ ]:

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
embedding_deployment = os.getenv("AZURE_OPENAI_EMBEDDING")
model = os.getenv("AZURE_OPENAI_DEPLOYMENT")
 token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)
 

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_version=api_version,
    azure_ad_token_provider=token_provider
)

print("Modules succesvol geladen!")


Modules succesvol geladen!


### Testing

In [ ]:
df_temp_WRV_testing = df_wrv_testing.copy()

df_temp_WRV_testing.rename(columns={"Octopus zaaknummer": "case_id", "Dictum": "dictum"}, inplace=True)

cols_to_clear = ["Legal_Text", "chunk_uitleg"]
for col in cols_to_clear:
    if col in df_temp_WRV_testing.columns:
        df_temp_WRV_testing[col] = ""


rag_system = SimpleVectorRag(cases=df_wrv_training)

def generate_juridische_beslissing_met_rag(zaak_id, reden_jurist=""):
    case_row = df_temp_WRV_testing[df_temp_WRV_testing['case_id'] == zaak_id]
    if case_row.empty:
        return f"Error: Case {zaak_id} not found."
    
    row = case_row.iloc[0]
    bezwaar = row.get("chunk_bezwaar", "")
    hoorzitting = row.get("chunk_hoorzitting", "")
    dictum = row.get("dictum", "")
    
    similar_cases = rag_system.retrieve(query=bezwaar, top_k=3, dictum=dictum)
    reference_context = ""
    for i, doc in enumerate(similar_cases, 1):
        reference_context += (
            f"--- REFERENCE CASE {i} ---\n"
            f"Wettelijk kader: {doc['chunk_wetten']}\n"
            f"Overwegingen: {doc['chunk_uitleg']}\n\n"
        )

    # Clean metadata for the prompt
    exclude = ["chunk_bezwaar", "chunk_hoorzitting", "chunk_uitleg", "Legal_Text", "chunk_wetten", "geanonimiseerd_doc_inhoud"]
    metadata = row.drop(labels=exclude, errors='ignore').to_dict()
    
    messages = [
        {"role": "system", "content": "You are a legal officer for the Municipality of Amsterdam. Write in Dutch."},
        {"role": "user", "content": f"""
        Draft a formal legal decision letter.
        
        CASE METADATA: {json.dumps(metadata, default=str)}
        
        CITIZEN OBJECTION: {bezwaar}
        HEARING INFO: {hoorzitting}
        JURIST REASONING: {reden_jurist}
        
        REQUIRED VERDICT: {dictum}
        
        REFERENCE LEGAL LOGIC:
        {reference_context}
        
        Draft the full letter including: 1. Inleiding, 2. Beslissing op bezwaar, 3. Bezwaren, 4. Hoorzitting, 5. Overwegingen, 6. Praktische gevolgen.
        """}
    ]
    
    response = client.chat.completions.create(
        model=os.getenv("AZURE_OPENAI_DEPLOYMENT"),
        messages=messages,
        temperature=0.3
    )
    
    return response.choices[0].message.content

zaak_id_input = case_id_test
reden_jurist_input = input("Voer de sturing voor de jurist in (optioneel):")

print("\nGenereren van brief...")
resultaat = generate_juridische_beslissing_met_rag(
    zaak_id=zaak_id_input,
    reden_jurist=reden_jurist_input
)

print("\n--- GEGENEREERDE BRIEF ---")
print(resultaat)


Genereren van brief...

--- GEGENEREERDE BRIEF ---
**Gemeente Amsterdam**  
Directie Wonen  
Postbus 95089  
1090 HB Amsterdam  

**Betreft:** Beslissing op bezwaar  
**Zaaknummer:** WO.25.013871.001  
**Datum verzendbesluit:** 11 december 2025  

**Geachte mevrouw [Achternaam],**  

Hierbij ontvangt u de beslissing op uw bezwaarschrift van 4 oktober 2025, gericht tegen het besluit van 1 oktober 2025 (kenmerk: F2025.31116). In dit besluit is vastgesteld dat er één zoekpunt is afgebouwd in WoningNet, omdat u in september 2025 niet heeft gereageerd op sociale huurwoningen via WoningNet of Woonmatch.  

Wij hebben uw bezwaar zorgvuldig beoordeeld en komen tot de volgende beslissing.  

---

### **1. Beslissing op bezwaar**  
Wij verklaren uw bezwaarschrift **ongegrond**. Dit betekent dat het besluit van 1 oktober 2025 in stand blijft. Het zoekpunt dat in september 2025 is afgebouwd, wordt niet hersteld.  

---

### **2. Bezwaren**  
In uw bezwaarschrift heeft u de volgende punten aangevo

In [ ]:
def get_case_info(zaak_id: str, df: pd.DataFrame):
    case_data = df[df['case_id'] == zaak_id]
    
    if case_data.empty:
        print(f"Case ID '{zaak_id}' not found in df_temp_WRV_testing.")
        return
    
    print(f"--- Information for Case: {zaak_id} ---")
    print(case_data.iloc[0].to_string())

    print(f"--- chunk_bezwaar ---")
    print(df_temp_WRV_testing["chunk_bezwaar"].iloc[0])
    print(f"--- chunk_hoorzitting---")
    print(df_temp_WRV_testing["chunk_hoorzitting"].iloc[0])


get_case_info(case_id_test, df_temp_WRV_testing)




--- Information for Case: WO.25.013871.001 ---
case_id                                                          WO.25.013871.001
Zaaktype                                                                  Bezwaar
Subtype                                                Point Sanction (Reduction)
Primair Besluit                                                       F2025.31116
Verzenddatum primair besluit                                  2025-10-01 00:00:00
dictum                                                                  ongegrond
Zaakstatus                                                            Afgehandeld
Behandelaar                                                           Kahveci, G.
Startdatum                                                    2025-10-04 00:00:00
Datum zitting                                                                 NaN
Verzendatum besluit                                           2025-12-11 00:00:00
Planningsdatum                                     

In [ ]:
def calculate_tfidf_similarity(text1: str, text2: str, rag_system) -> float:
    vec1 = rag_system._build_tfidf_vector(Counter(_tokenize(str(text1))))
    vec2 = rag_system._build_tfidf_vector(Counter(_tokenize(str(text2))))
    
    all_tokens = set(vec1.keys()) | set(vec2.keys())

    dot_product = sum(vec1.get(t, 0.0) * vec2.get(t, 0.0) for t in all_tokens)
    
    mag1 = math.sqrt(sum(v * v for v in vec1.values()))
    mag2 = math.sqrt(sum(v * v for v in vec2.values()))
    
    if mag1 * mag2 == 0:
        return 0.0
        
    return dot_product / (mag1 * mag2)




original_doc = df_wrv_testing[df_wrv_testing['case_id'] == target_case_test]['Legal_Text'].iloc[0]

score = calculate_tfidf_similarity(resultaat, original_doc, rag_system)
print(f"TF-IDF Similarity Score: {score:.4f}")

TF-IDF Similarity Score: 0.6081


## LLM + Legal Bert

In [ ]:
endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_version = os.getenv("AZURE_OPENAI_API_VERSION")
model_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default"
)

client = AzureOpenAI(
    azure_endpoint=endpoint,
    api_version=api_version,
    azure_ad_token_provider=token_provider
)

df_temp_WRV_testing = df_wrv_testing.copy()

cols_to_drop = ["Legal_Text", "chunk_uitleg"]
df_temp_WRV_testing.drop(columns=[c for c in cols_to_drop if c in df_temp_WRV_testing.columns], inplace=True)


def generate_juridische_beslissing_met_rag(zaak_id: str, reden_jurist: str = ""):
    row = df_temp_WRV_testing[df_temp_WRV_testing["case_id"] == zaak_id]
    if row.empty: 
        return "Error: Case ID not found."
    
    row = row.iloc[0]
    bezwaar = row.get("chunk_bezwaar", "")
    hoorzitting = row.get("chunk_hoorzitting", "")
    dictum = row.get("dictum", "")
    
    similar_cases = rag_system_bert.retrieve(query=bezwaar, top_k=3, dictum=dictum)
    
    reference_context = "\n".join([
        f"--- REF {i} ---\nWettelijk kader: {c['chunk_wetten']}\nOverwegingen: {c['chunk_uitleg']}"
        for i, c in enumerate(similar_cases, 1)
    ])
    
    exclude_keys = ["chunk_bezwaar", "chunk_hoorzitting", "dictum"]
    meta = row.drop(labels=exclude_keys, errors="ignore").to_dict()
    
    messages = [
        {"role": "system", "content": "You are a legal officer for Amsterdam. Write in Dutch."},
        {"role": "user", "content": f"""
        Draft a formal legal decision letter.
        
        Metadata: {json.dumps(meta, default=str)}
        Objection: {bezwaar}
        Hearing: {hoorzitting}
        Jurist Note: {reden_jurist}
        Verdict: {dictum}
        
        Reference logic (laws and reasoning): 
        {reference_context}
        
        Draft the full letter including: 1. Inleiding, 2. Beslissing op bezwaar, 3. Bezwaren, 4. Hoorzitting, 5. Overwegingen, 6. Praktische gevolgen.
        """}
    ]
    
    response = client.chat.completions.create(
        model=model_deployment,
        messages=messages,
        temperature=0.3
    )
    return response.choices[0].message.content



In [ ]:
def calculate_bert_similarity(generated_text: str, original_text: str, rag_model) -> float:
    embeddings = rag_model.model.encode([generated_text, original_text], convert_to_tensor=True)
    return util.cos_sim(embeddings[0], embeddings[1]).item()

zaak_id_test = case_id_test
sturing = input("Enter manual jurist notes (optional): ")

print("\nGenerating letter...")
generated_letter = generate_juridische_beslissing_met_rag(zaak_id_test, sturing)

print("\n" + "="*50)
print("GENERATED LETTER")
print("="*50)
print(generated_letter)

original_doc = df_wrv_testing[df_wrv_testing['case_id'] == zaak_id_test]['Legal_Text'].iloc[0]
score = calculate_bert_similarity(generated_letter, original_doc, rag_system_bert)

print("\n" + "="*50)
print(f"LegalBERT Similarity Score: {score:.4f}")
print("="*50)


Generating letter...

GENERATED LETTER
**Gemeente Amsterdam**  
Directie Wonen  
[Adres]  
[Postcode en Plaats]  

**Datum:** 11 december 2025  
**Kenmerk:** WO.25.013871.001  
**Onderwerp:** Beslissing op bezwaar inzake afbouw zoekpunt  

**Geachte heer/mevrouw,**

Hierbij ontvangt u de beslissing op uw bezwaar van 4 oktober 2025 tegen het besluit van 1 oktober 2025, waarbij één zoekpunt is afgebouwd in het systeem van WoningNet. Wij hebben uw bezwaar zorgvuldig beoordeeld en informeren u hierbij over onze beslissing en de overwegingen die daaraan ten grondslag liggen.

---

### **1. Inleiding**  
Op 1 oktober 2025 is er een besluit genomen om één zoekpunt af te bouwen in uw WoningNet-account, omdat u in de maand september 2025 niet heeft gereageerd op sociale huurwoningen via WoningNet of Woonmatch. U heeft op 4 oktober 2025 bezwaar gemaakt tegen dit besluit. Uw bezwaar is geregistreerd onder zaaknummer WO.25.013871.001 en behandeld door een jurist van het Juridisch Bureau van de ge

## Evaluation

In [ ]:

print("Loading model")
try:
    nl_evaluator_model
except NameError:
    nl_evaluator_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


def generate_vector_rag(zaak_id, reden_jurist=""):
    case_row = df_temp_WRV_testing[df_temp_WRV_testing['case_id'] == zaak_id]
    if case_row.empty: return f"Error: Case {zaak_id} not found.", []
    row = case_row.iloc[0]
    bezwaar, hoorzitting, dictum = row.get("chunk_bezwaar", ""), row.get("chunk_hoorzitting", ""), row.get("dictum", "")
    
    similar_cases = rag_system.retrieve(query=bezwaar, top_k=3, dictum=dictum)
    reference_context = "\n".join([f"--- REF {i} ---\nWettelijk kader: {c['chunk_wetten']}\nOverwegingen: {c['chunk_uitleg']}\n" for i, c in enumerate(similar_cases, 1)])
    
    exclude = ["chunk_bezwaar", "chunk_hoorzitting", "chunk_uitleg", "Legal_Text", "chunk_wetten", "geanonimiseerd_doc_inhoud"]
    metadata = row.drop(labels=exclude, errors='ignore').to_dict()
    
    messages = [
        {"role": "system", "content": "You are a legal officer for the Municipality of Amsterdam. Write in Dutch."},
        {"role": "user", "content": f"Draft a formal legal decision letter.\n\nCASE METADATA: {json.dumps(metadata, default=str)}\nCITIZEN OBJECTION: {bezwaar}\nHEARING INFO: {hoorzitting}\nJURIST REASONING: {reden_jurist}\nREQUIRED VERDICT: {dictum}\nREFERENCE LEGAL LOGIC:\n{reference_context}\nDraft the full letter including: 1. Inleiding, 2. Beslissing op bezwaar, 3. Bezwaren, 4. Hoorzitting, 5. Overwegingen, 6. Praktische gevolgen."}
    ]
    response = client.chat.completions.create(model=os.getenv("AZURE_OPENAI_DEPLOYMENT"), messages=messages, temperature=0.3)
    
    return response.choices[0].message.content, similar_cases


def generate_bert_rag(zaak_id, reden_jurist=""):
    case_row = df_temp_WRV_testing[df_temp_WRV_testing["case_id"] == zaak_id]
    if case_row.empty: return "Error: Case ID not found.", []
    row = case_row.iloc[0]
    bezwaar, hoorzitting, dictum = row.get("chunk_bezwaar", ""), row.get("chunk_hoorzitting", ""), row.get("dictum", "")
    
    similar_cases = rag_system_bert.retrieve(query=bezwaar, top_k=3, dictum=dictum)
    reference_context = "\n".join([f"--- REF {i} ---\nWettelijk kader: {c['chunk_wetten']}\nOverwegingen: {c['chunk_uitleg']}" for i, c in enumerate(similar_cases, 1)])
    
    exclude = ["chunk_bezwaar", "chunk_hoorzitting", "dictum"]
    metadata = row.drop(labels=exclude, errors="ignore").to_dict()
    
    messages = [
        {"role": "system", "content": "You are a legal officer for Amsterdam. Write in Dutch."},
        {"role": "user", "content": f"Draft a formal legal decision letter.\n\nMetadata: {json.dumps(metadata, default=str)}\nObjection: {bezwaar}\nHearing: {hoorzitting}\nJurist Note: {reden_jurist}\nVerdict: {dictum}\nReference logic (laws and reasoning): \n{reference_context}\nDraft the full letter including: 1. Inleiding, 2. Beslissing op bezwaar, 3. Bezwaren, 4. Hoorzitting, 5. Overwegingen, 6. Praktische gevolgen."}
    ]
    response = client.chat.completions.create(model=model_deployment, messages=messages, temperature=0.3)
    
    return response.choices[0].message.content, similar_cases


def evaluate_rag_context(retrieved_cases, original_uitleg):
    if not retrieved_cases or not str(original_uitleg).strip(): return 0.0
    scores = []
    for case in retrieved_cases:
        embeddings = nl_evaluator_model.encode([str(original_uitleg), str(case["chunk_uitleg"])], convert_to_tensor=True)
        scores.append(util.cos_sim(embeddings[0], embeddings[1]).item())
    return sum(scores) / len(scores)

def evaluate_letter(generated_text, original_text, tfidf_system):
    gen_str, orig_str = str(generated_text), str(original_text)
    
    vec1 = tfidf_system._build_tfidf_vector(Counter(_tokenize(gen_str)))
    vec2 = tfidf_system._build_tfidf_vector(Counter(_tokenize(orig_str)))
    all_tokens = set(vec1.keys()) | set(vec2.keys())
    dot = sum(vec1.get(t, 0.0) * vec2.get(t, 0.0) for t in all_tokens)
    mag1, mag2 = math.sqrt(sum(v*v for v in vec1.values())), math.sqrt(sum(v*v for v in vec2.values()))
    tfidf_score = (dot / (mag1 * mag2)) if (mag1 * mag2) > 0 else 0.0

    embeddings = nl_evaluator_model.encode([gen_str, orig_str], convert_to_tensor=True)
    bert_score = util.cos_sim(embeddings[0], embeddings[1]).item()
    
    return tfidf_score, bert_score


zaak_id_test = "WO.23.004999.001"
sturing = input("Voer de sturing voor de jurist in (optioneel): ")

print(f"\n[1/2] Genereren met TF-IDF Vector RAG...")
vector_letter, vector_cases = generate_vector_rag(zaak_id_test, sturing)

print(f"[2/2] Genereren met LegalBERT RAG...")
bert_letter, bert_cases = generate_bert_rag(zaak_id_test, sturing)

original_row = df_wrv_testing[df_wrv_testing['case_id'] == zaak_id_test].iloc[0]
original_letter = original_row['Legal_Text']
original_uitleg = original_row['chunk_uitleg']

vector_rag_score = evaluate_rag_context(vector_cases, original_uitleg)
bert_rag_score = evaluate_rag_context(bert_cases, original_uitleg)

vector_tfidf_score, vector_bert_score = evaluate_letter(vector_letter, original_letter, rag_system)
bert_tfidf_score, bert_bert_score = evaluate_letter(bert_letter, original_letter, rag_system)


print("\n" + "="*60)
print("VECTOR RAG vs. BERT RAG 🏆")
print("="*60)
print(f"Zaak ID: {zaak_id_test}")

print("\n--- 1. RAG Quality ---")
print("Semantic Match with created 'uitleg' and actual 'uitleg'.")
print(f"  TF-IDF Vector RAG score : {vector_rag_score:.4f}")
print(f"  LegalBERT RAG score     : {bert_rag_score:.4f}")

print("\n--- 2. Quality Generated Letter (Lexical) ---")
print("TF-IDF score: Did the LLM use the same words")
print(f"  LLM Vector RAG      : {vector_tfidf_score:.4f}")
print(f"  LLM BERT RAG        : {bert_tfidf_score:.4f}")

print("\n--- 3. Quality Generated Letter (Semantic) ---")
print("Does the LLM letter convey the same meaning as the original?")
print(f"  LLM Vector RAG      : {vector_bert_score:.4f}")
print(f"  LLM BERT RAG        : {bert_bert_score:.4f}")
print("="*60)

Loading model



[1/2] Genereren met TF-IDF Vector RAG...
[2/2] Genereren met LegalBERT RAG...

VECTOR RAG vs. BERT RAG 🏆
Zaak ID: WO.23.004999.001

--- 1. RAG Quality ---
Semantic Match with created 'uitleg' and actual 'uitleg'.
  TF-IDF Vector RAG score : 0.6699
  LegalBERT RAG score     : 0.5760

--- 2. Quality Generated Letter (Lexical) ---
TF-IDF score: Did the LLM use the same words
  LLM Vector RAG      : 0.5332
  LLM BERT RAG        : 0.6081

--- 3. Quality Generated Letter (Semantic) ---
Does the LLM letter convey the same meaning as the original?
  LLM Vector RAG      : 0.5915
  LLM BERT RAG        : 0.6345
